# Dataloader Strategy Comparison
**Goal:** Compare different data loading strategies and their impact on train/val distribution.

## Strategies to Compare
| Strategy | File | Description |
|----------|------|-------------|
| V1 (Original) | `data_loading.py` | Random split - potential leakage |
| V2 (Temporal) | `data_loadingV2.py` | Temporal stratified - last 15% per band |
| V3 (Random Stratified) | `data_loadingV3_random.py` | Random stratified by band |
| V4 (File-based) | `data_loadingV4_file.py` | Single file as validation |
| Bins (Weighted) | `data_loadingbins.py` | Weighted sampling for balance |

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Path setup
PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.band_mapping import load_band_mapping, classify_band
from utils.reproducibility_utils import seed_everything

SEED = 42
seed_everything(SEED)

TRAIN_PATH = PROJECT_ROOT / "datasets_challenge" / "train"
band_dict = load_band_mapping()

print(f"Project root: {PROJECT_ROOT}")

## 1. Load Data with Each Strategy

In [ ]:
def extract_targets(loader):
    """Extract all targets from a dataloader."""
    targets = []
    for X, y in loader:
        targets.extend(y.numpy())
    return np.array(targets)

def get_distribution(targets, band_dict):
    """Get band distribution from targets."""
    bands = classify_band(targets, band_dict)
    total = len(bands)
    return {
        'many': 100 * (bands == 'many').sum() / total,
        'medium': 100 * (bands == 'medium').sum() / total,
        'few': 100 * (bands == 'few').sum() / total,
        'count': total
    }

results = {}

### Strategy V2: Temporal Stratified Split

In [ ]:
from utils.data_loadingV2 import load_data as load_data_v2

print("Loading with V2 (Temporal Stratified)...")
train_v2, val_v2, scaler_v2, _, _ = load_data_v2(
    train_path=str(TRAIN_PATH), val_ratio=0.15,
    columns_to_drop=['sideSlip', 'vyCG', 'longitude', 'latitude', 'time'],
    sequence_length=20, target="sideSlip", threshold=True, threshold_value=3.0,
    threshold_column="vxCG", batchsize=64, scaling=True, seed=SEED, shuffle=False
)

train_targets_v2 = extract_targets(train_v2)
val_targets_v2 = extract_targets(val_v2)

results['V2_Temporal'] = {
    'train': get_distribution(train_targets_v2, band_dict),
    'val': get_distribution(val_targets_v2, band_dict)
}
print(f"Train: {len(train_targets_v2):,} | Val: {len(val_targets_v2):,}")

### Strategy V3: Random Stratified Split

In [ ]:
from utils.data_loadingV3_random import load_data as load_data_v3

print("\nLoading with V3 (Random Stratified)...")
train_v3, val_v3, scaler_v3, _, _ = load_data_v3(
    train_path=str(TRAIN_PATH), val_ratio=0.15,
    columns_to_drop=['sideSlip', 'vyCG', 'longitude', 'latitude', 'time'],
    sequence_length=20, target="sideSlip", threshold=True, threshold_value=3.0,
    threshold_column="vxCG", batchsize=64, scaling=True, seed=SEED, shuffle=False
)

train_targets_v3 = extract_targets(train_v3)
val_targets_v3 = extract_targets(val_v3)

results['V3_Random'] = {
    'train': get_distribution(train_targets_v3, band_dict),
    'val': get_distribution(val_targets_v3, band_dict)
}
print(f"Train: {len(train_targets_v3):,} | Val: {len(val_targets_v3):,}")

### Strategy V4: File-Based Split

In [ ]:
from utils.data_loadingV4_file import load_data as load_data_v4

print("\nLoading with V4 (File-Based, file 7)...")
train_v4, val_v4, scaler_v4, _, _ = load_data_v4(
    train_path=str(TRAIN_PATH), val_file_num=7,
    columns_to_drop=['sideSlip', 'vyCG', 'longitude', 'latitude', 'time'],
    sequence_length=20, target="sideSlip", threshold=True, threshold_value=3.0,
    threshold_column="vxCG", batchsize=64, scaling=True, seed=SEED, shuffle=False
)

train_targets_v4 = extract_targets(train_v4)
val_targets_v4 = extract_targets(val_v4)

results['V4_FileBased'] = {
    'train': get_distribution(train_targets_v4, band_dict),
    'val': get_distribution(val_targets_v4, band_dict)
}
print(f"Train: {len(train_targets_v4):,} | Val: {len(val_targets_v4):,}")

## 2. Distribution Comparison

In [ ]:
# Create comparison DataFrame
comparison_data = []
for strategy, data in results.items():
    comparison_data.append({
        'Strategy': strategy,
        'Split': 'Train',
        'Many (%)': data['train']['many'],
        'Medium (%)': data['train']['medium'],
        'Few (%)': data['train']['few'],
        'Count': data['train']['count']
    })
    comparison_data.append({
        'Strategy': strategy,
        'Split': 'Val',
        'Many (%)': data['val']['many'],
        'Medium (%)': data['val']['medium'],
        'Few (%)': data['val']['few'],
        'Count': data['val']['count']
    })

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

In [ ]:
# Visualize distribution comparison
fig = make_subplots(rows=1, cols=3, subplot_titles=['Many-Shot %', 'Medium-Shot %', 'Few-Shot %'])

strategies = list(results.keys())
colors = {'Train': 'blue', 'Val': 'red'}

for col, band in enumerate(['Many (%)', 'Medium (%)', 'Few (%)'], 1):
    for split in ['Train', 'Val']:
        subset = comparison_df[comparison_df['Split'] == split]
        fig.add_trace(
            go.Bar(x=subset['Strategy'], y=subset[band], name=split if col == 1 else None,
                   marker_color=colors[split], showlegend=(col == 1)),
            row=1, col=col
        )

fig.update_layout(title='Train vs Val Distribution by Strategy', height=400, barmode='group')
fig.show()

## 3. Distribution Mismatch Analysis

In [ ]:
# Calculate distribution mismatch for each strategy
mismatch_data = []
for strategy, data in results.items():
    many_diff = abs(data['train']['many'] - data['val']['many'])
    medium_diff = abs(data['train']['medium'] - data['val']['medium'])
    few_diff = abs(data['train']['few'] - data['val']['few'])
    total_mismatch = many_diff + medium_diff + few_diff
    
    mismatch_data.append({
        'Strategy': strategy,
        'Many Δ': many_diff,
        'Medium Δ': medium_diff,
        'Few Δ': few_diff,
        'Total Mismatch': total_mismatch
    })

mismatch_df = pd.DataFrame(mismatch_data).sort_values('Total Mismatch')
print("Distribution Mismatch (|Train - Val|):")
print(mismatch_df.to_string(index=False))

In [ ]:
# Visualize mismatch
fig = go.Figure(go.Bar(
    x=mismatch_df['Strategy'],
    y=mismatch_df['Total Mismatch'],
    marker_color=['green' if x < 5 else 'orange' if x < 10 else 'red' 
                  for x in mismatch_df['Total Mismatch']],
    text=[f"{x:.1f}" for x in mismatch_df['Total Mismatch']],
    textposition='outside'
))
fig.update_layout(
    title='Train/Val Distribution Mismatch by Strategy (Lower = Better)',
    xaxis_title='Strategy',
    yaxis_title='Total Mismatch (%)',
    height=400
)
fig.show()

## 4. Target Distribution Histograms

In [ ]:
# Compare target distributions side by side
fig = make_subplots(rows=len(results), cols=2, 
                    subplot_titles=[f'{s} Train' if i%2==0 else f'{s} Val' 
                                    for s in results.keys() for i in range(2)])

row = 1
for strategy in results.keys():
    # Get targets
    if strategy == 'V2_Temporal':
        train_t, val_t = train_targets_v2, val_targets_v2
    elif strategy == 'V3_Random':
        train_t, val_t = train_targets_v3, val_targets_v3
    else:
        train_t, val_t = train_targets_v4, val_targets_v4
    
    fig.add_trace(go.Histogram(x=train_t, nbinsx=100, name=f'{strategy} Train', 
                               marker_color='blue', showlegend=False), row=row, col=1)
    fig.add_trace(go.Histogram(x=val_t, nbinsx=100, name=f'{strategy} Val',
                               marker_color='red', showlegend=False), row=row, col=2)
    row += 1

fig.update_layout(height=300*len(results), title='Target Distribution Comparison')
fig.show()

## 5. Sequence Overlap Analysis (Data Leakage Check)

In [ ]:
# Check if there's temporal overlap between train and val
# This is important for V1 (random split) which can cause leakage

print("Sequence Overlap Analysis:")
print("-" * 50)

# For V2 (Temporal), train and val should be from different time periods
print("V2 (Temporal): Uses last 15% of each band's groups")
print("  → No temporal overlap by design")

# For V3 (Random), groups are randomly shuffled
print("\nV3 (Random): Random stratified split of groups")
print("  → No sequence overlap (groups kept intact)")

# For V4 (File-based), entire file is held out
print("\nV4 (File-based): Entire file held out for validation")
print("  → No overlap (completely separate file)")

## 6. Group-Level Analysis

In [ ]:
# Analyze number of groups in each split
print("Group-Level Statistics:")
print("-" * 50)

# V2 statistics
print(f"\nV2 (Temporal):")
print(f"  Train sequences: {len(train_targets_v2):,}")
print(f"  Val sequences: {len(val_targets_v2):,}")
print(f"  Val ratio: {len(val_targets_v2) / (len(train_targets_v2) + len(val_targets_v2)) * 100:.1f}%")

# V3 statistics
print(f"\nV3 (Random):")
print(f"  Train sequences: {len(train_targets_v3):,}")
print(f"  Val sequences: {len(val_targets_v3):,}")
print(f"  Val ratio: {len(val_targets_v3) / (len(train_targets_v3) + len(val_targets_v3)) * 100:.1f}%")

# V4 statistics
print(f"\nV4 (File-based):")
print(f"  Train sequences: {len(train_targets_v4):,}")
print(f"  Val sequences: {len(val_targets_v4):,}")
print(f"  Val ratio: {len(val_targets_v4) / (len(train_targets_v4) + len(val_targets_v4)) * 100:.1f}%")

## Summary

| Strategy | Pros | Cons | Use Case |
|----------|------|------|----------|
| V2 (Temporal) | No leakage, preserves time order | May have distribution shift | When temporal patterns matter |
| V3 (Random) | Best distribution match | Loses temporal info | When distribution match is critical |
| V4 (File-based) | Simplest, clearly separate | File may not represent overall | Quick experiments |

### Recommendation
- **For this competition**: V2 (Temporal) is recommended because:
  1. Preserves temporal ordering within groups
  2. Stratified by band ensures all classes represented
  3. No data leakage between train/val
  4. Best matches how test data would be collected